
# Tuned 2048 DQN Notebook

Notebook này là bản nâng cấp từ `DQN_for_2048_game.ipynb`, đã tích hợp các thay đổi được đề xuất để cải thiện:

- **maximum tile reached**
- **fraction of illegal actions avoided** *(đo theo raw argmax trước khi mask để metric có ý nghĩa)*
- **average greedy return over multiple seeds**

Các thay đổi chính:

1. **Observation wrapper tốt hơn**: encode board thành tensor **one-hot 4x4x16** + scalar feature maps.
2. **CNN + dueling heads** để khai thác cấu trúc không gian của board.
3. **Double DQN** để giảm overestimation.
4. **Huber loss** thay cho MSE.
5. **Đánh giá nhiều seed** và báo cáo đầy đủ các metric thầy yêu cầu.
6. Có sẵn **reward-mode toggle** để so sánh `delta_return`, `state_rewards`, hoặc `shaped` reward.

> Gợi ý sử dụng: chạy notebook với `REWARD_MODE="delta_return"` trước. Sau đó mới thử `state_rewards` và `shaped` để làm ablation.


In [ ]:

# Colab / notebook setup
!python -V
!pip -q install --upgrade pip
!pip -q install open-spiel torch matplotlib pandas imageio tqdm


In [ ]:

import copy
import math
import random
import re
from collections import Counter, deque, namedtuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import pyspiel

print("PyTorch version:", torch.__version__)
print("OpenSpiel version:", pyspiel.__version__ if hasattr(pyspiel, "__version__") else "unknown")
print("CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE



## 1. Game inspection and robust helpers

OpenSpiel 2048 có chance nodes, nên wrapper sẽ auto-resolve các node ngẫu nhiên. Phần helper dưới đây cũng cố tình **robust** để notebook dễ chạy trên Colab hơn.


In [ ]:
game = pyspiel.load_game("2048")
state = game.new_initial_state()

print("Registered game:", game.get_type().short_name)
print("Num distinct actions:", game.num_distinct_actions())
print("Observation tensor shape:", game.observation_tensor_shape())
print("Observation tensor size:", game.observation_tensor_size())
print("Max game length:", game.max_game_length())
print("Initial state:\n", state)

In [ ]:

def extract_obs_grid(state, player_id=0):
    """Return a 4x4 float32 board from OpenSpiel observation/state."""
    for fn_name, args in [
        ("observation_tensor", (player_id,)),
        ("observation_tensor", tuple()),
        ("information_state_tensor", (player_id,)),
        ("information_state_tensor", tuple()),
    ]:
        fn = getattr(state, fn_name, None)
        if fn is None:
            continue
        try:
            obs = fn(*args)
            obs = np.asarray(obs, dtype=np.float32)
            if obs.shape == (4, 4):
                return obs
            if obs.size == 16:
                return obs.reshape(4, 4)
        except TypeError:
            pass

    numbers = [int(x) for x in re.findall(r"-?\d+", str(state))]
    if len(numbers) >= 16:
        return np.asarray(numbers[:16], dtype=np.float32).reshape(4, 4)

    raise RuntimeError("Could not extract a 4x4 board from the OpenSpiel state.")


def get_state_return(state):
    returns_fn = getattr(state, "returns", None)
    if returns_fn is None:
        return 0.0
    try:
        vals = returns_fn()
    except TypeError:
        vals = returns_fn
    vals = np.asarray(vals, dtype=np.float32).reshape(-1)
    return float(vals[0]) if vals.size else 0.0


def get_state_reward(state):
    rewards_fn = getattr(state, "rewards", None)
    if rewards_fn is None:
        return 0.0
    try:
        vals = rewards_fn()
    except TypeError:
        vals = rewards_fn
    vals = np.asarray(vals, dtype=np.float32).reshape(-1)
    return float(vals[0]) if vals.size else 0.0


def auto_resolve_chance_nodes(state, rng):
    while state.is_chance_node():
        outcomes = state.chance_outcomes()
        actions = np.asarray([a for a, _ in outcomes], dtype=np.int64)
        probs = np.asarray([p for _, p in outcomes], dtype=np.float64)
        probs = probs / probs.sum()
        chosen = int(rng.choice(actions, p=probs))
        state.apply_action(chosen)


def board_to_log2(board):
    board = np.asarray(board, dtype=np.float32)
    out = np.zeros_like(board, dtype=np.int64)
    mask = board > 0
    out[mask] = np.log2(board[mask]).astype(np.int64)
    return out


def monotonicity_score(log_board):
    log_board = np.asarray(log_board, dtype=np.int64)
    total = 0
    for arr in list(log_board) + list(log_board.T):
        arr = np.asarray(arr)
        non_inc = np.sum(arr[:-1] >= arr[1:])
        non_dec = np.sum(arr[:-1] <= arr[1:])
        total += max(int(non_inc), int(non_dec))
    return float(total) / 24.0  # 8 lines * 3 adjacent comparisons


def smoothness_score(log_board):
    log_board = np.asarray(log_board, dtype=np.int64)
    diffs = []
    for r in range(4):
        for c in range(4):
            if log_board[r, c] == 0:
                continue
            if r + 1 < 4 and log_board[r + 1, c] > 0:
                diffs.append(abs(int(log_board[r, c]) - int(log_board[r + 1, c])))
            if c + 1 < 4 and log_board[r, c + 1] > 0:
                diffs.append(abs(int(log_board[r, c]) - int(log_board[r, c + 1])))
    if not diffs:
        return 1.0
    return float(np.clip(1.0 - np.mean(diffs) / 15.0, 0.0, 1.0))


def encode_board_features(board, max_log2_channel=15):
    """
    Encode board as:
      - 16 one-hot channels for values {empty, 2, 4, ..., 32768}
      - 5 scalar feature maps: empty_count, max_tile_log2, corner_flag, monotonicity, smoothness
    Output shape: [21, 4, 4]
    """
    board = np.asarray(board, dtype=np.float32).reshape(4, 4)
    log_board = board_to_log2(board)
    clipped = np.clip(log_board, 0, max_log2_channel)

    one_hot = np.eye(max_log2_channel + 1, dtype=np.float32)[clipped]      # [4, 4, 16]
    one_hot = np.transpose(one_hot, (2, 0, 1))                             # [16, 4, 4]

    empty_frac = float(np.mean(board == 0))
    max_log = float(np.max(clipped) / max_log2_channel) if max_log2_channel > 0 else 0.0
    max_tile = np.max(board)
    corners = [board[0, 0], board[0, 3], board[3, 0], board[3, 3]]
    corner_flag = float(max_tile > 0 and max_tile in corners)
    mono = monotonicity_score(clipped)
    smooth = smoothness_score(clipped)

    scalars = [empty_frac, max_log, corner_flag, mono, smooth]
    scalar_maps = np.stack([
        np.full((4, 4), fill_value=v, dtype=np.float32) for v in scalars
    ], axis=0)

    return np.concatenate([one_hot, scalar_maps], axis=0).astype(np.float32)


def make_legal_mask(num_actions, legal_actions_list):
    mask = np.zeros(num_actions, dtype=np.float32)
    if legal_actions_list:
        mask[np.asarray(legal_actions_list, dtype=np.int64)] = 1.0
    return mask



## 2. Tuned OpenSpiel 2048 wrapper

Wrapper này hỗ trợ 3 chế độ reward để bạn có thể làm ablation:

- `delta_return`: reward chuẩn, sạch nhất để bắt đầu
- `state_rewards`: dùng `state.rewards()`
- `shaped`: thêm bonus nhẹ cho **ô trống** và **new max tile**

Theo khuyến nghị ban đầu, nên chạy `delta_return` trước.


In [ ]:

class OpenSpiel2048FeatureEnv:
    def __init__(
        self,
        seed=42,
        reward_mode="delta_return",
        empty_cell_bonus=0.02,
        new_max_tile_bonus=0.10,
    ):
        self.game = pyspiel.load_game("2048")
        self.player_id = 0
        self.num_actions = self.game.num_distinct_actions()
        self.rng = np.random.default_rng(seed)
        self.reward_mode = reward_mode
        self.empty_cell_bonus = float(empty_cell_bonus)
        self.new_max_tile_bonus = float(new_max_tile_bonus)
        self.state = None

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.state = self.game.new_initial_state()
        auto_resolve_chance_nodes(self.state, self.rng)
        board = extract_obs_grid(self.state, self.player_id)
        return encode_board_features(board)

    def legal_actions(self):
        if self.state is None or self.state.is_terminal():
            return []
        return list(self.state.legal_actions())

    def step(self, action):
        if self.state is None:
            raise RuntimeError("Call reset() before step().")
        legal = self.legal_actions()
        if action not in legal:
            raise ValueError(f"Illegal action {action}. Legal actions are {legal}")

        prev_board = extract_obs_grid(self.state, self.player_id)
        prev_empty = int(np.sum(prev_board == 0))
        prev_max_tile = int(np.max(prev_board))
        prev_return = get_state_return(self.state)

        self.state.apply_action(int(action))
        auto_resolve_chance_nodes(self.state, self.rng)

        board = extract_obs_grid(self.state, self.player_id)
        done = self.state.is_terminal()
        total_return = get_state_return(self.state)
        delta_return = float(total_return - prev_return)
        raw_state_reward = get_state_reward(self.state)
        delta_empty = int(np.sum(board == 0)) - prev_empty
        new_max_tile = float(np.max(board) > prev_max_tile)

        if self.reward_mode == "delta_return":
            reward = delta_return
        elif self.reward_mode == "state_rewards":
            reward = raw_state_reward
        elif self.reward_mode == "shaped":
            reward = (
                delta_return
                + self.empty_cell_bonus * delta_empty
                + self.new_max_tile_bonus * new_max_tile
            )
        else:
            raise ValueError(f"Unknown reward_mode: {self.reward_mode}")

        info = {
            "board": board.astype(np.int64),
            "state_text": str(self.state),
            "legal_actions": self.legal_actions(),
            "delta_return": delta_return,
            "raw_state_reward": raw_state_reward,
            "delta_empty": delta_empty,
            "new_max_tile_increased": bool(new_max_tile),
            "max_tile": int(np.max(board)),
            "total_return": total_return,
        }
        return encode_board_features(board), float(reward), done, info

    def render(self):
        if self.state is None:
            print("<empty env>")
        else:
            print(self.state)


In [ ]:

# Quick sanity check
sanity_env = OpenSpiel2048FeatureEnv(seed=123, reward_mode="delta_return")
obs = sanity_env.reset(seed=123)
print("Encoded obs shape:", obs.shape)
print("Initial legal actions:", sanity_env.legal_actions())
sanity_env.render()



## 3. Replay buffer and tuned Q-network

Mạng dưới đây kết hợp:

- **CNN** để dùng cấu trúc 2D của board
- **dueling heads** *(toggle được)*
- đầu vào là tensor `[channels, 4, 4]`


In [ ]:

Transition = namedtuple(
    "Transition",
    ["obs", "action", "reward", "next_obs", "done", "legal_mask", "next_legal_mask"],
)


class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def __len__(self):
        return len(self.buffer)

    def add(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))


class DuelingConvQNet(nn.Module):
    def __init__(self, in_channels, num_actions, dueling=True):
        super().__init__()
        self.dueling = dueling
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.trunk = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
        )
        if dueling:
            self.value_stream = nn.Sequential(
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Linear(128, 1),
            )
            self.adv_stream = nn.Sequential(
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Linear(128, num_actions),
            )
        else:
            self.head = nn.Sequential(
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Linear(128, num_actions),
            )

    def forward(self, x):
        z = self.features(x)
        z = self.trunk(z)
        if self.dueling:
            value = self.value_stream(z)
            adv = self.adv_stream(z)
            return value + adv - adv.mean(dim=1, keepdim=True)
        return self.head(z)



## 4. Central config

Các config này là bản “tuned” theo hướng đã đề xuất. Nếu muốn debug nhanh, giảm `NUM_EPISODES` xuống trước khi chạy full.


In [ ]:

CONFIG = {
    # Repro / training
    "TRAIN_SEED": 7,
    "NUM_EPISODES": 3000,
    "BUFFER_SIZE": 100_000,
    "BATCH_SIZE": 128,
    "GAMMA": 0.99,
    "LR": 2.5e-4,
    "TARGET_SYNC_EVERY": 1000,
    "LEARN_START": 5000,
    "LEARN_EVERY": 4,
    "MAX_STEPS_PER_EPISODE": 5000,
    "GRAD_CLIP": 10.0,

    # Exploration
    "EPS_START": 1.0,
    "EPS_END": 0.01,
    "EPS_DECAY_STEPS": 100_000,

    # Architecture / algorithm
    "USE_DOUBLE_DQN": True,
    "USE_DUELING": True,

    # Reward-mode ablation
    "REWARD_MODE": "delta_return",     # choose: delta_return, state_rewards, shaped
    "EMPTY_CELL_BONUS": 0.02,
    "NEW_MAX_TILE_BONUS": 0.10,

    # Probing during training
    "PROBE_EVERY_EPISODES": 100,
    "PROBE_SEEDS": [1001, 1002, 1003, 1004, 1005],

    # Final evaluation
    "EVAL_SEEDS": list(range(2001, 2051)),

    # Save
    "SAVE_PATH": "dqn_2048_tuned_cnn_double_dueling.pt",
}

CONFIG



## 5. Action selection, Double DQN update, and evaluation helpers

Điểm mới quan trọng ở đây:

- **Double DQN** trong phần target update
- **Huber loss** (`smooth_l1_loss`)
- metric **fraction of illegal actions avoided** được đo bằng:
  - lấy `raw argmax` từ Q-values **trước khi mask**
  - xem raw argmax đó có thuộc legal actions không


In [ ]:

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)



def epsilon_by_step(step, cfg):
    frac = min(1.0, step / cfg["EPS_DECAY_STEPS"])
    return cfg["EPS_START"] + frac * (cfg["EPS_END"] - cfg["EPS_START"])



def masked_greedy_action(q_net, obs, legal_actions, num_actions, epsilon=0.0, device=DEVICE, return_debug=False):
    if len(legal_actions) == 0:
        raise RuntimeError("No legal actions available.")

    with torch.no_grad():
        x = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        q_values = q_net(x).squeeze(0).detach().cpu().numpy()

    raw_argmax = int(np.argmax(q_values))
    raw_argmax_is_legal = raw_argmax in set(legal_actions)

    if random.random() < epsilon:
        action = int(random.choice(legal_actions))
    else:
        masked_q = q_values.copy()
        illegal = np.ones(num_actions, dtype=bool)
        illegal[np.asarray(legal_actions, dtype=np.int64)] = False
        masked_q[illegal] = -1e9
        action = int(np.argmax(masked_q))

    if return_debug:
        return action, {
            "raw_q": q_values,
            "raw_argmax": raw_argmax,
            "raw_argmax_is_legal": raw_argmax_is_legal,
        }
    return action



def dqn_update(batch, q_net, target_net, optimizer, cfg):
    obs = torch.tensor(np.asarray(batch.obs), dtype=torch.float32, device=DEVICE)
    actions = torch.tensor(batch.action, dtype=torch.int64, device=DEVICE).unsqueeze(1)
    rewards = torch.tensor(batch.reward, dtype=torch.float32, device=DEVICE)
    next_obs = torch.tensor(np.asarray(batch.next_obs), dtype=torch.float32, device=DEVICE)
    dones = torch.tensor(batch.done, dtype=torch.float32, device=DEVICE)
    next_legal_mask = torch.tensor(np.asarray(batch.next_legal_mask), dtype=torch.bool, device=DEVICE)

    q_values = q_net(obs)
    q_sa = q_values.gather(1, actions).squeeze(1)

    with torch.no_grad():
        no_legal_next = ~next_legal_mask.any(dim=1)

        if cfg["USE_DOUBLE_DQN"]:
            online_next_q = q_net(next_obs).masked_fill(~next_legal_mask, -1e9)
            next_actions = online_next_q.argmax(dim=1, keepdim=True)
            target_next_q = target_net(next_obs).gather(1, next_actions).squeeze(1)
            target_next_q = target_next_q.masked_fill(no_legal_next, 0.0)
        else:
            target_next_q = target_net(next_obs).masked_fill(~next_legal_mask, -1e9)
            target_next_q = target_next_q.max(dim=1).values
            target_next_q = target_next_q.masked_fill(no_legal_next, 0.0)

        target = rewards + (1.0 - dones) * cfg["GAMMA"] * target_next_q

    loss = F.smooth_l1_loss(q_sa, target)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(q_net.parameters(), cfg["GRAD_CLIP"])
    optimizer.step()
    return float(loss.item())



def evaluate_greedy_policy(
    q_net,
    seeds,
    cfg,
    reward_mode=None,
    render_rollout_seed=None,
):
    reward_mode = reward_mode or cfg["REWARD_MODE"]
    per_seed = []
    tile_counter = Counter()
    raw_argmax_legal_steps = 0
    total_decision_steps = 0
    rollout_trace = None

    for seed in seeds:
        env = OpenSpiel2048FeatureEnv(
            seed=seed,
            reward_mode=reward_mode,
            empty_cell_bonus=cfg["EMPTY_CELL_BONUS"],
            new_max_tile_bonus=cfg["NEW_MAX_TILE_BONUS"],
        )
        obs = env.reset(seed=seed)
        done = False
        total_reward = 0.0
        steps = 0
        max_tile = 0
        local_trace = []

        while not done and steps < cfg["MAX_STEPS_PER_EPISODE"]:
            legal = env.legal_actions()
            action, dbg = masked_greedy_action(
                q_net,
                obs,
                legal,
                env.num_actions,
                epsilon=0.0,
                return_debug=True,
                device=DEVICE,
            )
            raw_argmax_legal_steps += int(dbg["raw_argmax_is_legal"])
            total_decision_steps += 1

            next_obs, reward, done, info = env.step(action)
            total_reward += reward
            steps += 1
            max_tile = max(max_tile, int(info["max_tile"]))

            if render_rollout_seed is not None and seed == render_rollout_seed:
                local_trace.append({
                    "action": int(action),
                    "reward": float(reward),
                    "board": info["board"].copy(),
                    "state_text": info["state_text"],
                    "legal_actions": list(legal),
                    "raw_argmax": int(dbg["raw_argmax"]),
                    "raw_argmax_is_legal": bool(dbg["raw_argmax_is_legal"]),
                })

            obs = next_obs

        per_seed.append({
            "seed": seed,
            "greedy_return": float(total_reward),
            "steps": int(steps),
            "max_tile": int(max_tile),
        })
        tile_counter[int(max_tile)] += 1
        if render_rollout_seed is not None and seed == render_rollout_seed:
            rollout_trace = local_trace

    df = pd.DataFrame(per_seed)
    summary = {
        "mean_greedy_return": float(df["greedy_return"].mean()),
        "std_greedy_return": float(df["greedy_return"].std(ddof=0)),
        "median_greedy_return": float(df["greedy_return"].median()),
        "mean_max_tile": float(df["max_tile"].mean()),
        "max_tile_distribution": dict(sorted(tile_counter.items())),
        "fraction_illegal_actions_avoided": float(raw_argmax_legal_steps / max(total_decision_steps, 1)),
        "num_eval_seeds": int(len(seeds)),
        "table": df.sort_values(["max_tile", "greedy_return"], ascending=False).reset_index(drop=True),
    }
    return summary, rollout_trace



## 6. Training loop

Notebook này mặc định train **một seed**, nhưng phần evaluation cuối dùng **nhiều seed** để báo cáo metric ổn định hơn. Nếu bạn muốn làm báo cáo chặt hơn nữa, có thể lặp `train_one_seed` cho nhiều training seeds và gom kết quả.


In [ ]:

def train_one_seed(cfg, train_seed=None):
    train_seed = cfg["TRAIN_SEED"] if train_seed is None else int(train_seed)
    set_all_seeds(train_seed)

    env = OpenSpiel2048FeatureEnv(
        seed=train_seed,
        reward_mode=cfg["REWARD_MODE"],
        empty_cell_bonus=cfg["EMPTY_CELL_BONUS"],
        new_max_tile_bonus=cfg["NEW_MAX_TILE_BONUS"],
    )

    sample_obs = env.reset(seed=train_seed)
    in_channels = int(sample_obs.shape[0])
    num_actions = int(env.num_actions)

    q_net = DuelingConvQNet(in_channels, num_actions, dueling=cfg["USE_DUELING"]).to(DEVICE)
    target_net = DuelingConvQNet(in_channels, num_actions, dueling=cfg["USE_DUELING"]).to(DEVICE)
    target_net.load_state_dict(q_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(q_net.parameters(), lr=cfg["LR"])
    replay = ReplayBuffer(cfg["BUFFER_SIZE"])

    episode_returns = []
    episode_lengths = []
    episode_max_tiles = []
    loss_history = []
    probe_records = []

    global_step = 0

    for episode in tqdm(range(1, cfg["NUM_EPISODES"] + 1), desc=f"Training seed {train_seed}"):
        obs = env.reset(seed=train_seed + episode)
        done = False
        ep_return = 0.0
        ep_len = 0
        ep_max_tile = 0

        while not done and ep_len < cfg["MAX_STEPS_PER_EPISODE"]:
            legal = env.legal_actions()
            legal_mask = make_legal_mask(num_actions, legal)
            eps = epsilon_by_step(global_step, cfg)
            action = masked_greedy_action(q_net, obs, legal, num_actions, epsilon=eps, device=DEVICE)

            next_obs, reward, done, info = env.step(action)
            next_legal = env.legal_actions() if not done else []
            next_legal_mask = make_legal_mask(num_actions, next_legal)

            replay.add(obs, action, reward, next_obs, done, legal_mask, next_legal_mask)

            obs = next_obs
            ep_return += reward
            ep_len += 1
            ep_max_tile = max(ep_max_tile, int(info["max_tile"]))
            global_step += 1

            if len(replay) >= max(cfg["BATCH_SIZE"], cfg["LEARN_START"]) and global_step % cfg["LEARN_EVERY"] == 0:
                batch = replay.sample(cfg["BATCH_SIZE"])
                loss = dqn_update(batch, q_net, target_net, optimizer, cfg)
                loss_history.append(loss)

            if global_step % cfg["TARGET_SYNC_EVERY"] == 0:
                target_net.load_state_dict(q_net.state_dict())

        episode_returns.append(float(ep_return))
        episode_lengths.append(int(ep_len))
        episode_max_tiles.append(int(ep_max_tile))

        if cfg["PROBE_EVERY_EPISODES"] and episode % cfg["PROBE_EVERY_EPISODES"] == 0:
            probe_summary, _ = evaluate_greedy_policy(
                q_net,
                seeds=cfg["PROBE_SEEDS"],
                cfg=cfg,
                reward_mode=cfg["REWARD_MODE"],
            )
            probe_records.append({
                "episode": episode,
                "mean_greedy_return": probe_summary["mean_greedy_return"],
                "mean_max_tile": probe_summary["mean_max_tile"],
                "fraction_illegal_actions_avoided": probe_summary["fraction_illegal_actions_avoided"],
            })

    final_summary, rollout_trace = evaluate_greedy_policy(
        q_net,
        seeds=cfg["EVAL_SEEDS"],
        cfg=cfg,
        reward_mode=cfg["REWARD_MODE"],
        render_rollout_seed=cfg["EVAL_SEEDS"][-1],
    )

    return {
        "train_seed": train_seed,
        "q_net": q_net,
        "target_net": target_net,
        "config": copy.deepcopy(cfg),
        "in_channels": in_channels,
        "num_actions": num_actions,
        "episode_returns": episode_returns,
        "episode_lengths": episode_lengths,
        "episode_max_tiles": episode_max_tiles,
        "loss_history": loss_history,
        "probe_df": pd.DataFrame(probe_records),
        "final_eval": final_summary,
        "rollout_trace": rollout_trace,
    }


In [ ]:

# Start training
run = train_one_seed(CONFIG)
print("Done. Final mean greedy return:", run["final_eval"]["mean_greedy_return"])
print("Final mean max tile:", run["final_eval"]["mean_max_tile"])
print("Fraction illegal actions avoided:", run["final_eval"]["fraction_illegal_actions_avoided"])



## 7. Visualize training and probe metrics


In [ ]:

def moving_average(x, w=20):
    x = np.asarray(x, dtype=np.float32)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w, dtype=np.float32) / w, mode="valid")


plt.figure(figsize=(16, 4))

plt.subplot(1, 4, 1)
plt.plot(run["episode_returns"], alpha=0.35, label="episode return")
ma = moving_average(run["episode_returns"], 20)
plt.plot(range(len(ma)), ma, label="moving avg (20)")
plt.title("Training return")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.legend()

plt.subplot(1, 4, 2)
plt.plot(run["episode_lengths"], alpha=0.5)
plt.title("Episode length")
plt.xlabel("Episode")
plt.ylabel("Steps")

plt.subplot(1, 4, 3)
plt.plot(run["episode_max_tiles"], alpha=0.5)
plt.title("Max tile per episode")
plt.xlabel("Episode")
plt.ylabel("Tile")

plt.subplot(1, 4, 4)
plt.plot(run["loss_history"], alpha=0.7)
plt.title("Huber TD loss")
plt.xlabel("Update step")
plt.ylabel("Loss")

plt.tight_layout()
plt.show()

if not run["probe_df"].empty:
    probe_df = run["probe_df"]
    plt.figure(figsize=(15, 4))

    plt.subplot(1, 3, 1)
    plt.plot(probe_df["episode"], probe_df["mean_greedy_return"], marker="o")
    plt.title("Probe mean greedy return")
    plt.xlabel("Episode")
    plt.ylabel("Return")

    plt.subplot(1, 3, 2)
    plt.plot(probe_df["episode"], probe_df["mean_max_tile"], marker="o")
    plt.title("Probe mean max tile")
    plt.xlabel("Episode")
    plt.ylabel("Tile")

    plt.subplot(1, 3, 3)
    plt.plot(probe_df["episode"], probe_df["fraction_illegal_actions_avoided"], marker="o")
    plt.title("Probe fraction illegal avoided")
    plt.xlabel("Episode")
    plt.ylabel("Fraction")

    plt.tight_layout()
    plt.show()



## 8. Final multi-seed evaluation

Đây là cell quan trọng nhất cho báo cáo vì nó trả đúng 3 metric thầy yêu cầu.


In [ ]:

final_eval = run["final_eval"]

print("Mean greedy return:", final_eval["mean_greedy_return"])
print("Std greedy return:", final_eval["std_greedy_return"])
print("Median greedy return:", final_eval["median_greedy_return"])
print("Mean max tile:", final_eval["mean_max_tile"])
print("Max tile distribution:", final_eval["max_tile_distribution"])
print("Fraction of illegal actions avoided:", final_eval["fraction_illegal_actions_avoided"])
print("Number of eval seeds:", final_eval["num_eval_seeds"])

display(final_eval["table"].head(15))


In [ ]:

# A compact summary table for reporting
report_df = pd.DataFrame([
    {
        "metric": "maximum tile reached (mean over eval seeds)",
        "value": final_eval["mean_max_tile"],
    },
    {
        "metric": "fraction of illegal actions avoided",
        "value": final_eval["fraction_illegal_actions_avoided"],
    },
    {
        "metric": "average greedy return over multiple seeds",
        "value": final_eval["mean_greedy_return"],
    },
])
report_df



## 9. Inspect one greedy rollout


In [ ]:

rollout = run["rollout_trace"] or []
print("Rollout length:", len(rollout))
if rollout:
    print("Final board:")
    print(rollout[-1]["board"])

n_show = min(5, len(rollout))
for i, step_info in enumerate(rollout[-n_show:], start=max(1, len(rollout) - n_show + 1)):
    print("=" * 70)
    print(
        f"Step {i} | action={step_info['action']} | raw_argmax={step_info['raw_argmax']} | "
        f"raw_argmax_is_legal={step_info['raw_argmax_is_legal']} | reward={step_info['reward']:.2f}"
    )
    print(step_info["board"])



## 10. Optional ablations

Nếu muốn viết báo cáo kỹ hơn, bạn có thể chạy các ablation sau:

1. `REWARD_MODE = "delta_return"`
2. `REWARD_MODE = "state_rewards"`
3. `REWARD_MODE = "shaped"`
4. `USE_DUELING = False`
5. giảm / tăng `EPS_DECAY_STEPS`, `LR`, `NUM_EPISODES`

Cell dưới đây giúp bạn chạy nhanh vài biến thể bằng cách copy config gốc.


In [ ]:

# Example ablation configs (disabled by default because training is expensive)
ABLATIONS = [
    {"name": "delta_return_baseline", "REWARD_MODE": "delta_return", "USE_DUELING": True},
    {"name": "state_rewards", "REWARD_MODE": "state_rewards", "USE_DUELING": True},
    {"name": "shaped_reward", "REWARD_MODE": "shaped", "USE_DUELING": True},
    {"name": "no_dueling", "REWARD_MODE": "delta_return", "USE_DUELING": False},
]

# To run ablations, uncomment below.
# ablation_rows = []
# for variant in ABLATIONS:
#     cfg = copy.deepcopy(CONFIG)
#     cfg.update({k: v for k, v in variant.items() if k != "name"})
#     result = train_one_seed(cfg, train_seed=cfg["TRAIN_SEED"])
#     ablation_rows.append({
#         "name": variant["name"],
#         "mean_greedy_return": result["final_eval"]["mean_greedy_return"],
#         "mean_max_tile": result["final_eval"]["mean_max_tile"],
#         "fraction_illegal_actions_avoided": result["final_eval"]["fraction_illegal_actions_avoided"],
#     })
# pd.DataFrame(ablation_rows).sort_values("mean_greedy_return", ascending=False)



## 11. Save checkpoint


In [ ]:

checkpoint = {
    "model_state_dict": run["q_net"].state_dict(),
    "target_state_dict": run["target_net"].state_dict(),
    "config": run["config"],
    "in_channels": run["in_channels"],
    "num_actions": run["num_actions"],
    "episode_returns": run["episode_returns"],
    "episode_lengths": run["episode_lengths"],
    "episode_max_tiles": run["episode_max_tiles"],
    "loss_history": run["loss_history"],
    "final_eval": {
        k: v for k, v in run["final_eval"].items() if k != "table"
    },
}

torch.save(checkpoint, CONFIG["SAVE_PATH"])
print("Saved checkpoint to:", CONFIG["SAVE_PATH"])



## 12. What this notebook already implements from the proposal

Notebook này đã gộp các ý đã đề xuất trước đó:

- observation wrapper tốt hơn
- CNN
- Double DQN
- dueling heads
- Huber loss
- reward-mode comparison hooks
- evaluation trên nhiều seed
- metric `fraction of illegal actions avoided` được đo đúng kiểu **pre-mask raw argmax**

Nếu bạn cần bản báo cáo đi kèm notebook, có thể dùng chính `report_df` và `final_eval["table"]` làm nguồn số liệu.
